Здесь представлен пайплайн обработки данных для исследования семантики мест в корпусе устных свидетельств (для 11 пилотных интервью).

1. Парсинг первичной тематической разметки с платформы «Сфира» АНО Научно-гуманитарный центр Сэфер.

2. Фильтрация фрагментов интервью, релевантных для анализа семантики мест.

3. Подготовка текстовых файлов для загрузки в QualCoder для вторичного кодирования, т.к. этот инструмент не принимает другой формат файлов.

4. Восстановление fragment_id после ручного семантического кодирования в QualCoder.

5. Соединение размеченных данных с реестром мест (который велся параллельно с вторичным семантическим кодированием) и добавление текста фрагментов для наглядности.

**Скрипт 1: Парсинг размеченных интервью с платформы «Сфира».**

Назначение: загружает страницы редактирования интервью по их ID, извлекает разбивку по минутам, тексты минутных фрагментов и назначенные им теги (ключевые слова, персоналии, географию). Сохраняет результат в parsed_interviews.csv.

Входные данные: список ID интервью, cookie-строка из браузера (необходима для доступа к страницам).

Выходные данные: parsed_interviews.csv – таблица с колонками: interview_id, interview_code, minute, text, keywords, personalities, geography, fragment_id.

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# Вставляем свою cookie-строку из браузера после авторизации на sfira.org
COOKIE_STRING = "_ga=GA1.1.884313440.1764920522; _identity=d1de8d078fefe1ce9a966f1bdf75d8ab8c8cc1c2b47080c4449d3fe3bb17ad97a%3A2%3A%7Bi%3A0%3Bs%3A9%3A%22_identity%22%3Bi%3A1%3Bs%3A47%3A%22%5B90%2C%22fDnERVyteCFTPL8mxSpEbr79NG_g3S_z%22%2C2592000%5D%22%3B%7D; _ga_2Y4HY49SE7=GS2.1.s1767633003$o5$g0$t1767633003$j60$l0$h0; PHPSESSID=agqup7eokqkljggib0800fvgfj; _csrf=60f9acc681f062e328c1224bc3397cc5fe5ff15243bc49eacda30f352500bfc4a%3A2%3A%7Bi%3A0%3Bs%3A5%3A%22_csrf%22%3Bi%3A1%3Bs%3A32%3A%22iGJbDM0OV4TNuzdvaMd8fPWeTqQQEx0q%22%3B%7D"
interview_ids = [1099, 1078, 1089, 1100, 1103, 1093, 1094, 1098, 1092, 1096, 1090]  # ваши ID

cookies = {}
for item in COOKIE_STRING.split('; '):
    if '=' in item:
        key, val = item.split('=', 1)
        cookies[key] = val

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

def fetch_interview_page(interview_id):
    url = f'https://sfira.org/backend/web/interview/update?id={interview_id}'
    try:
        response = requests.get(url, headers=HEADERS, cookies=cookies, timeout=15)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Ошибка загрузки интервью {interview_id}: {e}")
        return None

def parse_interview(html, interview_id):
    soup = BeautifulSoup(html, 'html.parser')
    title_input = soup.find('input', {'id': 'interview-title'}) or soup.find('input', {'name': 'Interview[title]'})
    interview_code = title_input.get('value').strip() if title_input else f"ID_{interview_id}"
    rows = soup.select('tr.multiple-input-list__item')
    if not rows:
        print(f"Интервью {interview_id}: не найдены строки с текстом.")
        return []
    data = []
    for row in rows:
        minute_cell = row.find('td', class_='list-cell__minute')
        minute = minute_cell.find('input').get('value') if minute_cell else None
        text_cell = row.find('td', class_='list-cell__text')
        textarea = text_cell.find('textarea') if text_cell else None
        text = textarea.get_text(strip=True) if textarea else None
        keywords = extract_tags(row, 'keywordsTags')
        personalities = extract_tags(row, 'personalitiesTags')
        geography = extract_tags(row, 'geographyTags')
        data.append({
            'minute': minute,
            'text': text,
            'keywords': ', '.join(keywords),
            'personalities': ', '.join(personalities),
            'geography': ', '.join(geography),
            'interview_code': interview_code,
            'interview_id': interview_id
        })
    return data

def extract_tags(row, tag_type):
    cell = row.find('td', class_=f'list-cell__{tag_type}')
    if not cell:
        return []
    select_tag = cell.find('select', {'name': re.compile(rf'Interview\[texts\]\[\d+\]\[{tag_type}\]')})
    if not select_tag:
        return []
    selected = select_tag.find_all('option', selected=True)
    tags = []
    for opt in selected:
        raw = opt.get_text()
        clean = raw.replace('\\u00a0', ' ').strip()
        clean = re.sub(r'^[\\s=–—-]+', '', clean).strip()
        clean = re.sub(r'^[.\\s]+', '', clean).strip()
        if clean:
            tags.append(clean)
    return tags

all_records = []
for idx, iid in enumerate(interview_ids, 1):
    print(f"[{idx}/{len(interview_ids)}] Обрабатываю интервью ID {iid}...")
    html = fetch_interview_page(iid)
    if html:
        records = parse_interview(html, iid)
        if records:
            all_records.extend(records)
            print(f"    -> найдено {len(records)} фрагментов")
        else:
            print(f"    -> нет текстовых фрагментов")
    else:
        print(f"    -> пропущено (ошибка загрузки)")
    time.sleep(1)

if all_records:
    df = pd.DataFrame(all_records)
    df = df[['interview_id', 'interview_code', 'minute', 'text', 'keywords', 'personalities', 'geography']]
    df['fragment_id'] = df['interview_id'].astype(str) + '_' + df['minute'].astype(str)
    df.to_csv('parsed_interviews.csv', index=False, encoding='utf-8')
    print(f"\n✅ Готово! Сохранено {len(df)} фрагментов из {len(interview_ids)} интервью.")
    print("Файл: parsed_interviews.csv")
else:
    print("❌ Не удалось собрать ни одного фрагмента.")

[1/11] Обрабатываю интервью ID 1099...
    -> найдено 12 фрагментов
[2/11] Обрабатываю интервью ID 1078...
    -> найдено 57 фрагментов
[3/11] Обрабатываю интервью ID 1089...
    -> найдено 67 фрагментов
[4/11] Обрабатываю интервью ID 1100...
    -> найдено 92 фрагментов
[5/11] Обрабатываю интервью ID 1103...
    -> найдено 66 фрагментов
[6/11] Обрабатываю интервью ID 1093...
    -> найдено 159 фрагментов
[7/11] Обрабатываю интервью ID 1094...
    -> найдено 118 фрагментов
[8/11] Обрабатываю интервью ID 1098...
    -> найдено 68 фрагментов
[9/11] Обрабатываю интервью ID 1092...
    -> найдено 70 фрагментов
[10/11] Обрабатываю интервью ID 1096...
    -> найдено 105 фрагментов
[11/11] Обрабатываю интервью ID 1090...
    -> найдено 29 фрагментов

✅ Готово! Сохранено 843 фрагментов из 11 интервью.
Файл: parsed_interviews.csv


**Скрипт 2: Фильтрация фрагментов для семантического кодирования**

Назначение: из всего массива данных отбираются только те фрагменты, которые содержат хотя бы один «пространственно-релевантный» тег из предопределённого списка. Результат сохраняется в filtered_grouped_for_coding.csv.

Входные данные: parsed_interviews.csv.

Выходные данные: filtered_grouped_for_coding.csv – таблица с колонками: fragment_id, interview_id, interview_code, minute, text, all_tags (объединённые релевантные теги).

In [8]:
import pandas as pd

df = pd.read_csv('/content/parsed_interviews.csv')

# Множество тегов, релевантных для анализа семантики мест
place_tags = {
    'Еврейское пространство', 'Локальный текст', 'Городской текст', 'Культурная жизнь',
    'Еврейский театр', 'Репрезентация еврейской истории в музее', 'Этно-конфессиональные фестивали',
    'Топонимика', 'Символика', 'Сенсорные образы', 'Туризм', 'Дом и быт', 'Кошерная посуда',
    'Мезуза', 'Перестройка дома', 'Убранство дома', 'Устройство дома', 'Местечко', 'Синагога',
    'Региональная идентичность', 'Численность евреев', 'Пища', 'Кошерная/некошерная еда',
    'Повседневная еда', 'Праздничная еда', 'Смешанные браки', 'Характер', 'Добрососедство',
    'Межэтнические конфликты', 'Этнические стереотипы', 'Кладбище', 'Могила', 'Погребальный обряд',
    'Поминальные дни', 'Похороны', 'Йом Кипур', 'Песах', 'Пурим', 'Рош-ха-Шана', 'Суббота',
    'Суккот', 'Ханука', 'Шавуот', 'Раввин', 'Молитва', 'Молитвенные принадлежности',
    'Ритуальное омовение', 'Рабанит', 'Вторая мировая война', 'Фронт', 'Тыл', 'Эвакуация',
    'Холокост', 'Мемориализация', 'Образование', 'Еврейские школы', 'Религиозное образование',
    'Семейная история', 'Происхождение фамилии', 'Профессия', 'Переезд', 'Советская еврейская история',
    'Биробиджанский проект', 'Первые переселенцы', 'Землячества', 'Предприятия региона',
    'Репрессии', 'Еврейские колхозы', 'Современная еврейская жизнь', 'Еврейские организации',
    'Общинная жизнь', 'Телевидение и пресса', 'Эмиграция', 'Репатриация в Израиль (алия)',
    'Легенды', 'Предания', 'Слухи и толки', 'Неевреи в синагоге', 'Неевреи на еврейском кладбище',
    'Еврейская речь', 'Идиш', 'Иврит', 'Языковая политика'
}

def extract_relevant_tags(row):
    tags = []
    if pd.notna(row['geography']):
        tags.extend([t.strip() for t in row['geography'].split(',') if t.strip()])
    if pd.notna(row['keywords']):
        tags.extend([t.strip() for t in row['keywords'].split(',') if t.strip()])
    if pd.notna(row['personalities']):
        tags.extend([t.strip() for t in row['personalities'].split(',') if t.strip()])
    return [tag for tag in tags if tag in place_tags]

df['relevant_tags'] = df.apply(extract_relevant_tags, axis=1)
df_filtered = df[df['relevant_tags'].apply(len) > 0].copy()
df_filtered['all_tags'] = df_filtered['relevant_tags'].apply(lambda x: '; '.join(x))

result = df_filtered[['fragment_id', 'interview_id', 'interview_code', 'minute', 'text', 'all_tags']]
result.to_csv('filtered_grouped_for_coding.csv', index=False, encoding='utf-8')
print(f'Сохранено фрагментов, подходящих для семантического кодирования: {len(result)}')

Сохранено фрагментов, подходящих для семантического кодирования: 781


**Скрипт 3: Подготовка текстовых файлов для QualCoder**

Назначение: преобразует отфильтрованную таблицу в набор текстовых файлов (по одному на интервью), которые можно загрузить в программу для качественного анализа QualCoder. Каждый файл содержит фрагменты с указанием fragment_id, минуты и исходных тегов.

Входные данные: filtered_grouped_for_coding.csv.

Выходные данные: папка taguette_input с файлами вида EAO_25_XX_YYY.txt.

In [9]:
import pandas as pd
import os

df = pd.read_csv('/content/filtered_grouped_for_coding.csv')
output_dir = 'taguette_input'
os.makedirs(output_dir, exist_ok=True)

for code, group in df.groupby('interview_code'):
    filename = f"{code}.txt".replace('/', '_').replace('\\', '_')
    with open(os.path.join(output_dir, filename), 'w', encoding='utf-8') as f:
        for _, row in group.iterrows():
            f.write(f"=== fragment_id: {row['fragment_id']} ===\n")
            f.write(f"minute: {row['minute']}\n")
            f.write(f"первичные_теги: {row['all_tags']}\n")
            f.write("текст:\n")
            f.write(row['text'].replace('\n', ' ').strip() + '\n\n')
    print(f"Создан файл: {output_dir}/{filename}")

Создан файл: taguette_input/EAO_25_11_Bb.txt
Создан файл: taguette_input/EAO_25_12_Bb.txt
Создан файл: taguette_input/EAO_25_19_Am.txt
Создан файл: taguette_input/EAO_25_22_Bb.txt
Создан файл: taguette_input/EAO_25_23_Bb.txt
Создан файл: taguette_input/EAO_25_29_Bb.txt
Создан файл: taguette_input/EAO_25_35_Bb.txt
Создан файл: taguette_input/EAO_25_40_Vald.txt
Создан файл: taguette_input/EAO_25_41_Vald.txt
Создан файл: taguette_input/EAO_25_42_Vald.txt
Создан файл: taguette_input/EAO_25_44_Vald.txt


**Скрипт 4: Восстановление fragment_id из разметки QualCoder**

Назначение: после того как мы вручную разметили фрагменты в QualCoder и экспортировали разметку в CSV (файл Разметка_пилот.csv), этот скрипт сопоставляет каждую размеченную строку с исходным fragment_id по совпадению текста и имени файла. Группирует коды и заметки по fragment_id и сохраняет результат в pilot_coded_grouped.csv.

Входные данные:



*   экспорт из QualCoder (например, Разметка_пилот.csv),
*   filtered_grouped_for_coding.csv.





Выходные данные: pilot_coded_grouped.csv – таблица с колонками: fragment_id, minute, File, codes, memos.

In [10]:
import pandas as pd

export = pd.read_csv('/content/Разметка_пилот.csv')
source = pd.read_csv('/content/filtered_grouped_for_coding.csv')
source['file_name'] = source['interview_code'] + '.txt'

def find_fragment(row, source_df):
    text = row['Coded']
    file = row['File']
    matches = source_df[(source_df['file_name'] == file) &
                        (source_df['text'].str.contains(text, na=False, regex=False))]
    if len(matches) == 1:
        return matches.iloc[0]['fragment_id'], matches.iloc[0]['minute']
    elif len(matches) > 1:
        print(f"Предупреждение: для текста '{text[:50]}...' найдено {len(matches)} совпадений")
        return matches.iloc[0]['fragment_id'], matches.iloc[0]['minute']
    else:
        return None, None

result = export.apply(lambda r: pd.Series(find_fragment(r, source)), axis=1)
result.columns = ['fragment_id', 'minute']
merged = pd.concat([export, result], axis=1)
merged = merged.dropna(subset=['fragment_id'])

grouped = merged.groupby(['fragment_id', 'minute', 'File']).agg({
    'Codename': lambda x: '; '.join(x),
    'Coded_Memo': lambda x: '; '.join([str(m) for m in x if pd.notna(m)])
}).reset_index()
grouped.rename(columns={'Codename': 'codes', 'Coded_Memo': 'memos'}, inplace=True)

grouped.to_csv('pilot_coded_grouped.csv', index=False)
print(f'Обработано фрагментов: {len(grouped)}')

Обработано фрагментов: 70


In [14]:
# Загружаем только что созданный файл
df = pd.read_csv('/content/pilot_coded_grouped.csv')

# Нормализуем fragment_id: добавляем '.0', если нет точки
df['fragment_id'] = df['fragment_id'].apply(lambda x: x if '.' in str(x) else str(x) + '.0')

# Сохраняем обратно
df.to_csv('/content/pilot_coded_grouped.csv', index=False, encoding='utf-8')

In [17]:
# Загружаем filtered_grouped_for_coding.csv
source_text = pd.read_csv('/content/filtered_grouped_for_coding.csv')

# Нормализуем fragment_id: добавляем '.0', если нет точки
source_text['fragment_id'] = source_text['fragment_id'].apply(lambda x: x if '.' in str(x) else str(x) + '.0')

# Сохраняем обратно
source_text.to_csv('/content/filtered_grouped_for_coding.csv', index=False, encoding='utf-8')

**Скрипт 5: Соединение с реестром мест и добавление текста**

Назначение: объединяет размеченные данные (pilot_coded_grouped.csv) с реестром мест (reestr_place.csv), где каждому месту сопоставлены fragment_id. Добавляет текст фрагментов из исходного файла filtered_grouped_for_coding.csv. Результат – полная таблица для анализа семантики мест.

Входные данные:



*   reestr_place.csv
*   pilot_coded_grouped.csv
*   filtered_grouped_for_coding.csv







Выходные данные: places_with_codes_and_text.csv – финальная таблица с колонками: place_id, name_place, fragment_id, minute, File, codes, memos, text.

In [20]:
import pandas as pd

# 1. Загружаем реестр мест
places = pd.read_csv('/content/reestr_place.csv', sep=';')
places = places.iloc[:, :3]

# 2. Загружаем размеченные данные (после восстановления fragment_id)
coded = pd.read_csv('/content/pilot_coded_grouped.csv')

# 3. Добавляем текст и первичные теги из исходного файла
source = pd.read_csv('/content/filtered_grouped_for_coding.csv')
# Берём уникальные fragment_id с текстом и тегами
source_unique = source[['fragment_id', 'text', 'all_tags']].drop_duplicates(subset=['fragment_id'])
# Присоединяем к coded
coded = coded.merge(source_unique, on='fragment_id', how='left')

# 4. Разворачиваем fragment_id в реестре мест
places['fragment_list'] = places['fragment_id'].str.split(',')
places_expanded = places.explode('fragment_list')
places_expanded['fragment_list'] = places_expanded['fragment_list'].str.strip()
places_expanded = places_expanded.drop(columns=['fragment_id'])
places_expanded = places_expanded.rename(columns={'fragment_list': 'fragment_id'})

# 5. Объединяем
merged = pd.merge(places_expanded, coded, on='fragment_id', how='left')

# 6. Сохраняем финальный результат
merged.to_csv('places_with_codes_and_text.csv', index=False, encoding='utf-8-sig')
print("✅ Файл places_with_codes_and_text.csv сохранён.")
print("Колонки:", merged.columns.tolist())

✅ Файл places_with_codes_and_text.csv успешно сохранён.


После выполнения всех пяти скриптов мы получили файл **places_with_codes_and_text.csv**, в котором для каждого места собраны все относящиеся к нему фрагменты интервью, семантические коды и заметки.

Это позволяет нам:

*   анализировать семантическую насыщенность каждого места и составлять его портрет

*   сравнивать, как одно и то же место наделяется различными смыслами разными информантами
*   анализировать соотношение тематической и семантической разметки

*   отбирать значимые точки для цифрового туристического маршрута

Все промежуточные файлы (parsed_interviews.csv, filtered_grouped_for_coding.csv, pilot_coded_grouped.csv) также сохраняются и могут быть использованы для дополнительных проверок или других аналитических задач.

In [23]:
semantic_place_pilot = pd.read_csv('/content/places_with_codes_and_text.csv')
semantic_place_pilot.head(10)

,place_id,name_place,fragment_id,minute,File,codes,memos,text,all_tags
0,bir,Биробиджан,1092_0.0,0.0,EAO_25_11_Bb.txt,place_institution; place_settlement; relation_...,place_id: kom_amur_ped_inst; place_id: bir,"Итак, с чего начнем? [Расскажите, пожалуйста, ...",Образование; Семейная история
1,bir,Биробиджан,1092_10.0,10.0,EAO_25_11_Bb.txt,place_settlement; relation_event,place_id: bir,"в Биробиджан, да, с пустыми руками, с пустыми ...",Семейная история; Идиш
2,bir,Биробиджан,1092_25.0,25.0,EAO_25_11_Bb.txt,place_infrastructure; place_public_space; plac...,place_id: big_gost_centr; place_id: bir,"приветствовать и говорить на идише, да? Посели...",Численность евреев; Биробиджанский проект; Идиш
3,bir,Биробиджан,1092_39.0,39.0,EAO_25_11_Bb.txt,emotion_negative; emotion_nostalgic; place_inf...,place_id: bir_vokzal; place_id: bir,"Богу. Ну, и третье, да, мы все, так сказать, з...",Локальный текст; Региональная идентичность; Эм...
4,bir,Биробиджан,1092_40.0,40.0,EAO_25_11_Bb.txt,memory_conflict; place_settlement; space_own_a...,place_id: bir,"Везде, по всему миру. Я часто бывал, когда был...",Семейная история; Эмиграция; Репатриация в Изр...
5,bir,Биробиджан,1092_41.0,41.0,EAO_25_11_Bb.txt,memory_material; memory_symbolic; place_memori...,place_id: bir_kladbiche; place_id: bir,"Где твои дети? В Австралии. А твои дети? Там, ...",Локальный текст; Кладбище; Семейная история; Э...
6,bir,Биробиджан,1089_16.0,16.0,EAO_25_19_Am.txt,narrative_crossing; place_settlement; relation...,"place_id: amurzet, place_id: bir",Исаака Лейбовича Кадемии. И наша бригада – кли...,Локальный текст; Топонимика; Семейная история
7,kom_amur_ped_inst,Педагогический институт в Комсомольске-на-Амуре,1092_0.0,0.0,EAO_25_11_Bb.txt,place_institution; place_settlement; relation_...,place_id: kom_amur_ped_inst; place_id: bir,"Итак, с чего начнем? [Расскажите, пожалуйста, ...",Образование; Семейная история
8,kom_amur_ped_inst,Педагогический институт в Комсомольске-на-Амуре,1092_36.0,36.0,EAO_25_11_Bb.txt,place_institution; relation_study,place_id: kom_amur_ped_inst,институт — сильный? Сильный. Хабаровский педаг...,Образование
9,bir_sinagoga,Синагога Бейт Тшува,1092_1.0,1.0,EAO_25_11_Bb.txt,place_work; process_interaction,place_id: bir_sinagoga,"заслуженный учитель Российской Федерации, поче...",Репрезентация еврейской истории в музее; Синаг...
